In [15]:
import polars as pl
import pandas as pd
import numpy as mp
import altair as alt
import pyarrow
from pyarrow import dictionary

from func.UsefulFunctions import *

This notebook is for merging the various group related data

In [16]:
groupDF = pd.read_csv("./Group Info/Query 114 - Musical Ensembles (and subclasses such as bands) + genre + type of group + inception year + country of origin + record label – Only group name is labelled.csv",low_memory=False)
groups_with_wikiDF = pd.read_csv("./Group Info/Query 18 - The Wikipedia Pages (if they have one) For All Music Ensembles.csv")
groups_with_MB_DF = pd.read_csv("./Group Info/Query 25 - Groups and their musicBrainzID.csv")
groups_with_statementsDF = pd.read_csv("./Group Info/Query 27 - Unlabelled Musical Groups and number of statements.csv")
groups_with_influencesDF = pd.read_csv("./Group Info/Query 115 - Groups and who they were influenced by.csv")

In [17]:
def addLabels(df, columnName):
    dictionary = getLabels(df[columnName].dropna().unique().tolist())
    labelLabel = columnName + "Label"
    dictionaryDF = pd.DataFrame(dictionary.items(), columns=[columnName, labelLabel])
    df = df.merge(dictionaryDF, on=columnName, how='left')

    return df


In [18]:
#Clean up each entity
groupDF[['group','genre','groupType','countryOfOrigin','recordLabel']] = groupDF[['group','genre','groupType','countryOfOrigin','recordLabel']].map(getEntityIDValue)
groupDF.inception = groupDF.inception.map(getYear)


# genre_dict = getLabels(groupDF['genre'].dropna().unique().tolist())
# genre_pdDF = pd.DataFrame(genre_dict.items(), columns=['genre','genreLabel'])
# groupDF = groupDF.merge(genre_pdDF, on='genre', how='left')
groupDF['genre'] = groupDF['genre'].where(groupDF['genre'].str.startswith('Q'), None)
groupDF = addLabels(groupDF, 'genre')
groupDF['groupType'] = groupDF['groupType'].where(groupDF['groupType'].str.startswith('Q'), None)
groupDF = addLabels(groupDF, 'groupType')
groupDF.countryOfOrigin = groupDF.countryOfOrigin.where(groupDF['countryOfOrigin'].str.startswith('Q'), None)
groupDF = addLabels(groupDF, 'countryOfOrigin')
groupDF['recordLabel'] = groupDF['recordLabel'].where(groupDF['recordLabel'].str.startswith('Q'), None)
groupDF = addLabels(groupDF, 'recordLabel')

groupDF.head(5)



The number of keys in the dictionary is 1705 
This is the list of ids: {wd:Q877693 wd:Q20474 wd:Q20502 wd:Q851213 wd:Q83270 wd:Q20378 wd:Q206159 wd:Q316930 wd:Q667659 wd:Q8341 wd:Q484641 wd:Q856872 wd:Q183504 wd:Q211573 wd:Q885561 wd:Q837837 wd:Q9759 wd:Q187760 wd:Q606393 wd:Q216961 wd:Q11366 wd:Q325504 wd:Q3071 wd:Q852767 wd:Q49451 wd:Q11403 wd:Q263734 wd:Q2389651 wd:Q208494 wd:Q28146601 wd:Q438503 wd:Q253918 wd:Q487914 wd:Q1425661 wd:Q850412 wd:Q2336819 wd:Q830325 wd:Q105527 wd:Q968730 wd:Q476983 wd:Q11399 wd:Q379671 wd:Q9730 wd:Q517415 wd:Q842324 wd:Q235858 wd:Q45981 wd:Q1298934 wd:Q2525544 wd:Q132438 wd:Q217467 wd:Q484179 wd:Q56351111 wd:Q486415 wd:Q1772422 wd:Q2045695 wd:Q6010 wd:Q37073 wd:Q186472 wd:Q11365 wd:Q1046768 wd:Q1391336 wd:Q822914 wd:Q1786724 wd:Q217191 wd:Q38848 wd:Q211756 wd:Q131578 wd:Q202930 wd:Q436016 wd:Q997481 wd:Q186170 wd:Q193355 wd:Q83440 wd:Q7749 wd:Q2265866 wd:Q372103 wd:Q208239 wd:Q959583 wd:Q617240 wd:Q483251 wd:Q483352 wd:Q9778 wd:Q753520 wd:Q76092 wd:Q48

,group,groupLabel,genre,groupType,inception,countryOfOrigin,recordLabel,genreLabel,groupTypeLabel,countryOfOriginLabel,recordLabelLabel
0,Q72092,Rage Against the Machine,Q877693,Q215380,2001.0,Q30,Q216364,rap rock,musical group,United States,Epic Records
1,Q72092,Rage Against the Machine,Q877693,Q215380,2001.0,Q30,Q1759336,rap rock,musical group,United States,Revelation Records
2,Q72349,Krewella,Q20474,Q14073567,2010.0,Q30,Q183387,dubstep,sibling duo,United States,Columbia Records
3,Q72349,Krewella,Q20474,Q14073567,2010.0,Q30,Q10398667,dubstep,sibling duo,United States,Monstercat
4,Q72349,Krewella,Q20502,Q14073567,2010.0,Q30,Q183387,house music,sibling duo,United States,Columbia Records


In [19]:
#This fixes any errors that were in the dataset by making sure the influenced by influence is an actual Q entity
groups_with_influencesDF[['group','influencedby']] = groups_with_influencesDF[['group','influencedby']].map(getEntityIDValue)
groups_with_influencesDF = groups_with_influencesDF[groups_with_influencesDF.influencedby.str.startswith('Q')]

groupDF = groupDF.merge(groups_with_influencesDF[['group','influencedby','influencedbyLabel']], how='left', on='group')


groupDF.head()

,group,groupLabel,genre,groupType,inception,countryOfOrigin,recordLabel,genreLabel,groupTypeLabel,countryOfOriginLabel,recordLabelLabel,influencedby,influencedbyLabel
0,Q72092,Rage Against the Machine,Q877693,Q215380,2001.0,Q30,Q216364,rap rock,musical group,United States,Epic Records,Q924413,24-7 Spyz
1,Q72092,Rage Against the Machine,Q877693,Q215380,2001.0,Q30,Q216364,rap rock,musical group,United States,Epic Records,Q392,Bob Dylan
2,Q72092,Rage Against the Machine,Q877693,Q215380,2001.0,Q30,Q216364,rap rock,musical group,United States,Epic Records,Q10708,Red Hot Chili Peppers
3,Q72092,Rage Against the Machine,Q877693,Q215380,2001.0,Q30,Q216364,rap rock,musical group,United States,Epic Records,Q622947,Jane's Addiction
4,Q72092,Rage Against the Machine,Q877693,Q215380,2001.0,Q30,Q216364,rap rock,musical group,United States,Epic Records,Q464476,Black Flag


In [20]:
display(groups_with_MB_DF[groups_with_MB_DF['musicBrainzID'].str.contains("http")])
individualWithIssue = groups_with_MB_DF[groups_with_MB_DF['musicBrainzID'].str.contains("http")]
individualWithIssue.to_csv("./missingValuesStore/groupEntityWithFormattingIssueInTheirMusicBrainz.csv", index=False)
groups_with_MB_DF[['group','musicBrainzID']] = groups_with_MB_DF[['group','musicBrainzID']].map(getEntityIDValue)
display(groups_with_MB_DF[['group','musicBrainzID']])
display(groups_with_MB_DF[groups_with_MB_DF.group == "Q140450482"])
groupDF = groupDF.merge(groups_with_MB_DF[['group','musicBrainzID']], how='left', on='group')

groupDF.head()

,group,groupLabel,musicBrainzID
72277,http://www.wikidata.org/entity/Q140450482,Q140450482,https://musicbrainz.org/artist/632cbe4d-5a1c-4...


,group,musicBrainzID
0,Q74160,7ba22880-978f-4e4c-849a-3baf167fc9c0
1,Q114543,e883133b-1b8e-4bfd-bc6f-45e1bf1584f0
2,Q116411,5f549372-3e08-42aa-ad67-0d3c60e1b86d
3,Q113484,0136756e-c79d-4d76-af7a-9471f0cca4e6
4,Q114651,3cc3942d-bbfc-446d-a8f4-f0e1e238ca86
...,...,...
77733,Q118294981,501997ee-e3c0-4d36-8bef-ccfc1c5d1635
77734,Q121603439,8d0ea88f-2820-4b61-a780-4406c9fb3e99
77735,Q115490619,6a052517-27f9-40bd-aab5-10f7f6ada4fa
77736,Q116368208,66ea2957-9423-46d0-b871-ef4afe5c1566


,group,groupLabel,musicBrainzID
72277,Q140450482,Q140450482,632cbe4d-5a1c-4e75-8276-e2e4e9faaa61


,group,groupLabel,genre,groupType,inception,countryOfOrigin,recordLabel,genreLabel,groupTypeLabel,countryOfOriginLabel,recordLabelLabel,influencedby,influencedbyLabel,musicBrainzID
0,Q72092,Rage Against the Machine,Q877693,Q215380,2001.0,Q30,Q216364,rap rock,musical group,United States,Epic Records,Q924413,24-7 Spyz,3798b104-01cb-484c-a3b0-56adc6399b80
1,Q72092,Rage Against the Machine,Q877693,Q215380,2001.0,Q30,Q216364,rap rock,musical group,United States,Epic Records,Q392,Bob Dylan,3798b104-01cb-484c-a3b0-56adc6399b80
2,Q72092,Rage Against the Machine,Q877693,Q215380,2001.0,Q30,Q216364,rap rock,musical group,United States,Epic Records,Q10708,Red Hot Chili Peppers,3798b104-01cb-484c-a3b0-56adc6399b80
3,Q72092,Rage Against the Machine,Q877693,Q215380,2001.0,Q30,Q216364,rap rock,musical group,United States,Epic Records,Q622947,Jane's Addiction,3798b104-01cb-484c-a3b0-56adc6399b80
4,Q72092,Rage Against the Machine,Q877693,Q215380,2001.0,Q30,Q216364,rap rock,musical group,United States,Epic Records,Q464476,Black Flag,3798b104-01cb-484c-a3b0-56adc6399b80


In [21]:
groups_with_wikiDF.group = groups_with_wikiDF.group.map(getEntityIDValue)
#This ensure it is a valid english wikipedia link
groups_with_wikiDF = groups_with_wikiDF[groups_with_wikiDF.article.str.contains("en.wikipedia")]
display(groups_with_wikiDF)
groups_with_statementsDF.group = groups_with_statementsDF.group.map(getEntityIDValue)

groupDF = groupDF.merge(groups_with_wikiDF[['group','article']], how='left', on='group')
groupDF = groupDF.merge(groups_with_statementsDF, how='left', on='group')

groupDF.head()

,group,groupLabel,page_titleEN,article
0,Q261,Linkin Park,Linkin Park,https://en.wikipedia.org/wiki/Linkin_Park
1,Q32645,Die Ärzte,Die Ärzte,https://en.wikipedia.org/wiki/Die_%C3%84rzte
2,Q32542,The Magnetic Fields,The Magnetic Fields,https://en.wikipedia.org/wiki/The_Magnetic_Fields
3,Q32543,Test Icicles,Test Icicles,https://en.wikipedia.org/wiki/Test_Icicles
4,Q32591,Crooked Still,Crooked Still,https://en.wikipedia.org/wiki/Crooked_Still
...,...,...,...,...
50443,Q112795795,18th & Addison,18th & Addison,https://en.wikipedia.org/wiki/18th_%26_Addison
50444,Q113133049,123 Andrés,123 Andrés,https://en.wikipedia.org/wiki/123_Andr%C3%A9s
50445,Q113022702,LF System,LF System,https://en.wikipedia.org/wiki/LF_System
50446,Q113022885,Frontperson,Frontperson,https://en.wikipedia.org/wiki/Frontperson


,group,groupLabel,genre,groupType,inception,countryOfOrigin,recordLabel,genreLabel,groupTypeLabel,countryOfOriginLabel,recordLabelLabel,influencedby,influencedbyLabel,musicBrainzID,article,statementCount
0,Q72092,Rage Against the Machine,Q877693,Q215380,2001.0,Q30,Q216364,rap rock,musical group,United States,Epic Records,Q924413,24-7 Spyz,3798b104-01cb-484c-a3b0-56adc6399b80,https://en.wikipedia.org/wiki/Rage_Against_the...,125.0
1,Q72092,Rage Against the Machine,Q877693,Q215380,2001.0,Q30,Q216364,rap rock,musical group,United States,Epic Records,Q392,Bob Dylan,3798b104-01cb-484c-a3b0-56adc6399b80,https://en.wikipedia.org/wiki/Rage_Against_the...,125.0
2,Q72092,Rage Against the Machine,Q877693,Q215380,2001.0,Q30,Q216364,rap rock,musical group,United States,Epic Records,Q10708,Red Hot Chili Peppers,3798b104-01cb-484c-a3b0-56adc6399b80,https://en.wikipedia.org/wiki/Rage_Against_the...,125.0
3,Q72092,Rage Against the Machine,Q877693,Q215380,2001.0,Q30,Q216364,rap rock,musical group,United States,Epic Records,Q622947,Jane's Addiction,3798b104-01cb-484c-a3b0-56adc6399b80,https://en.wikipedia.org/wiki/Rage_Against_the...,125.0
4,Q72092,Rage Against the Machine,Q877693,Q215380,2001.0,Q30,Q216364,rap rock,musical group,United States,Epic Records,Q464476,Black Flag,3798b104-01cb-484c-a3b0-56adc6399b80,https://en.wikipedia.org/wiki/Rage_Against_the...,125.0


In [22]:
groupDF[groupDF['genreLabel'].str.contains("wikidata")]

,group,groupLabel,genre,groupType,inception,countryOfOrigin,recordLabel,genreLabel,groupTypeLabel,countryOfOriginLabel,recordLabelLabel,influencedby,influencedbyLabel,musicBrainzID,article,statementCount


In [23]:
groupDF.dtypes

group                       str
groupLabel                  str
genre                       str
groupType                   str
inception               float64
countryOfOrigin             str
recordLabel                 str
genreLabel                  str
groupTypeLabel              str
countryOfOriginLabel        str
recordLabelLabel            str
influencedby                str
influencedbyLabel           str
musicBrainzID               str
article                     str
statementCount          float64
dtype: object

In [24]:
#de-duplicate the data and then save it as a parquet for cleaning later

groupDF.drop_duplicates(inplace=True)
groupDF.to_parquet("./Data/pipelinedGroupsWikidata.parquet")